# Train MNIST → integer-quantized FC stack (Keras, parametric)

**Approach (mirrors `plaintext_training1.ipynb`):**

1. Train a **continuous** Keras MLP with `hard_sigmoid` hidden activations on bipolarized inputs.
2. **Post-hoc discretize** every weight via `round(w · τ) / τ`.
3. Build a **strict-sign** evaluation model (real `tf.sign` activation) on top of the discretized weights and report the cleartext accuracy that the C++ FHE inference path will reproduce.
4. Export `round(W · τ)` and `round(b · τ)` as headerless integer CSVs (`fmt='%d'`) with the per-Linear-layer L1 / L2 norms — the message-space bound and the noise-growth factor for OpenFHE.

**Parametric vs `plaintext_training1.ipynb`:** topology, hidden-layer activation, `τ`, and training hyper-parameters all live in `CONFIG`. Setting `topology = [784, 30, 10]` reproduces the original DiNN-30; setting `topology = [784, 100, 10]` gives DiNN-100; `topology = [784, 100, 30, 10]` stacks two hidden layers; etc.


## 1. Dependencies

In [1]:
# The container image already ships tensorflow-cpu; this is a no-op when
# the package is installed and the only safety net when running outside.
%pip install -q tensorflow numpy


Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Everything tunable lives in `CONFIG`. The default reproduces `plaintext_training1.ipynb`'s `784 → 30 → 10` setup for MNIST; change `topology` to a longer list to add more hidden layers, bump `tau` to widen the integer dynamic range, etc.

In [40]:
import os
import datetime
import json as _json
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import mnist as _ds

print("tensorflow:", tf.__version__)

CONFIG = {
    "dataset":       "MNIST",
    "topology":      [784, 100, 100, 10],   # [in_dim, hidden..., n_classes]
    "hidden_activation": "hard_sigmoid",  # continuous training proxy for sign(x)
    "tau":           10,             # discretization scale: round(w * tau) / tau
    "epochs":        20,
    "batch_size":    128,
    "validation_split": 0.1,
    "binarize_threshold": 128,
    "seed":          0,
    "prefix":        "mnist",
    "output_dir":    "weights/mnist_dinn30",
}

assert CONFIG["topology"][0]  == 784, "MNIST input must be 784."
assert CONFIG["topology"][-1] == 10,   "MNIST output must be 10 classes."
assert len(CONFIG["topology"]) >= 2, "topology must list at least input + output dim."

tf.keras.utils.set_random_seed(CONFIG["seed"])
OUTPUT_DIR = Path(CONFIG["output_dir"])
print("Topology  :", " -> ".join(map(str, CONFIG["topology"])),
      f"({len(CONFIG['topology']) - 1} Dense layer(s))")
print("Output dir:", OUTPUT_DIR.resolve())


tensorflow: 2.21.0
Topology  : 784 -> 100 -> 100 -> 10 (3 Dense layer(s))
Output dir: /workspace/notebooks/weights/mnist_dinn30


## 3. Load MNIST and bipolarize the inputs

Each pixel byte is thresholded at 128 → `+1` if `pixel >= 128`, else `-1`, then the 28×28 grid is flattened in row-major order to length 784. This matches `io::LoadImageBipolar(path, 784, /*channels=*/1)` on the C++ side bit-for-bit.

In [41]:
(x_train, y_train), (x_test, y_test) = _ds.load_data()
print("raw x_train shape:", x_train.shape, " raw x_test shape:", x_test.shape)

def binarize(images, threshold=128):
    """Threshold each grayscale byte at `threshold` → {-1, +1},
    then flatten 28x28 to length 784 (row-major, matching the C++ side)."""
    bipolar = np.where(images < threshold, -1, 1)
    return bipolar.reshape(len(images), -1).astype(np.float32)

x_train_bin = binarize(x_train, threshold=CONFIG["binarize_threshold"])
x_test_bin  = binarize(x_test,  threshold=CONFIG["binarize_threshold"])

assert x_train_bin.shape[1] == 784, x_train_bin.shape
assert x_test_bin.shape[1]  == 784, x_test_bin.shape
assert set(np.unique(x_train_bin).tolist()).issubset({-1.0, 1.0})
assert set(np.unique(x_test_bin).tolist()).issubset({-1.0, 1.0})

print("x_train_bin:", x_train_bin.shape, x_train_bin.dtype,
      "min/max =", x_train_bin.min(), x_train_bin.max())
print("x_test_bin :", x_test_bin.shape,  x_test_bin.dtype,
      "min/max =", x_test_bin.min(),  x_test_bin.max())


raw x_train shape: (60000, 28, 28)  raw x_test shape: (10000, 28, 28)
x_train_bin: (60000, 784) float32 min/max = -1.0 1.0
x_test_bin : (10000, 784) float32 min/max = -1.0 1.0


## 4. Build the continuous model (parametric)

Generalization of `plaintext_training1.ipynb`'s `build_model(hidden_neurons=30)`: every entry of `topology[1:-1]` is a hidden `Dense` layer with the configured `hidden_activation`; the final `topology[-1]` is the softmax output. With the default `topology = [784, 30, 10]` this is identical to the original notebook (one `hard_sigmoid` hidden layer); `[784, 100, 30, 10]` would stack two.

In [42]:
def build_model(topology, hidden_activation="hard_sigmoid"):
    """Continuous Keras MLP. Hidden Denses use `hidden_activation`; output is softmax.

    The continuous activation is a smooth proxy for the strict `sign(x)` we will
    swap in at evaluation time after discretization. `hard_sigmoid` is the choice
    from `plaintext_training1.ipynb`; `tanh` and `sigmoid` also work and can be
    set via `CONFIG['hidden_activation']`."""
    in_dim      = topology[0]
    hidden_dims = topology[1:-1]
    out_dim     = topology[-1]

    layers = [keras.layers.Input(shape=(in_dim,))]
    for h in hidden_dims:
        layers.append(keras.layers.Dense(h, activation=hidden_activation))
    layers.append(keras.layers.Dense(out_dim, activation="softmax"))

    model = keras.Sequential(layers)
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_model(CONFIG["topology"], CONFIG["hidden_activation"])
model.summary()


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_30 (Dense)                │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 89,610 (350.04 KB)

 Trainable params: 89,610 (350.04 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Train the continuous model

In [43]:
history = model.fit(
    x_train_bin, y_train,
    epochs=CONFIG["epochs"],
    batch_size=CONFIG["batch_size"],
    validation_split=CONFIG["validation_split"],
)


Epoch 1/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7923 - loss: 0.8328 - val_accuracy: 0.9227 - val_loss: 0.2960
Epoch 2/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9126 - loss: 0.3021 - val_accuracy: 0.9428 - val_loss: 0.2100
Epoch 3/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9318 - loss: 0.2323 - val_accuracy: 0.9535 - val_loss: 0.1729
Epoch 4/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9445 - loss: 0.1887 - val_accuracy: 0.9590 - val_loss: 0.1511
Epoch 5/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9541 - loss: 0.1567 - val_accuracy: 0.9610 - val_loss: 0.1361
Epoch 6/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9615 - loss: 0.1325 - val_accuracy: 0.9648 - val_loss: 0.1257
Epoch 7/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9674 - loss: 0.1138 - val_accuracy: 0.9652 - val_loss: 0.1191
Epoch 8/20
422/422 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9722 - loss: 0.0982 - val_accuracy: 0.

## 6. Post-hoc discretize the weights

`process_weight(w) = round(w · τ) / τ` snaps each weight to the nearest 1/τ grid point. We then push the discretized weights back into the model (so the same Keras object can be re-evaluated), and we *also* keep the **pure-integer** copy `round(w · τ)` separately — those are what we ship to the C++ side as CSVs in § 8.

In [44]:
def process_weight(w, tau):
    """round(w * tau) / tau — snap to the nearest 1/tau lattice point."""
    return np.round(w * tau) / tau

def discretize_in_place(model, tau):
    """Replace every weight tensor with its discretized version, in place."""
    new_weights = [process_weight(w, tau) for w in model.get_weights()]
    model.set_weights(new_weights)
    return model

tau = CONFIG["tau"]
model = discretize_in_place(model, tau)

# Pure integer copies, paired (Wi, bi) for each Dense layer.
raw = model.get_weights()
assert len(raw) % 2 == 0, "Expected (W, b) pairs — got an odd number of tensors."
Ws_int = [np.round(raw[i]   * tau).astype(np.int64) for i in range(0, len(raw), 2)]
bs_int = [np.round(raw[i+1] * tau).astype(np.int64) for i in range(0, len(raw), 2)]

for i, (W, b) in enumerate(zip(Ws_int, bs_int), start=1):
    print(f"  Dense {i}: W{i} shape={W.shape}  range=[{int(W.min())}, {int(W.max())}]   "
          f"b{i} shape={b.shape}  range=[{int(b.min())}, {int(b.max())}]")


  Dense 1: W1 shape=(784, 100)  range=[-11, 9]   b1 shape=(100,)  range=[-1, 1]
  Dense 2: W2 shape=(100, 100)  range=[-16, 17]   b2 shape=(100,)  range=[-1, 1]
  Dense 3: W3 shape=(100, 10)  range=[-13, 9]   b3 shape=(10,)  range=[-1, 1]


## 7. Build a strict-`sign` evaluation model and measure cleartext accuracy

The continuous model used `hard_sigmoid` so gradients could flow during training; the deployed FHE path uses the exact `sign(x)` look-up table. We rebuild the network with `tf.sign` between Dense layers and copy the discretized weights in. The accuracy printed here is the cleartext accuracy the C++ FHE pipeline will reproduce on the same inputs.

In [45]:
def sign_activation(x):
    """Strict mathematical sign(x) — returns -1, 0, or +1."""
    return tf.sign(x)

def build_strict_sign_eval(weights, topology):
    """Same topology as build_model() but with tf.sign between Dense layers and a
    linear (no-softmax) output. argmax over the output logits is the prediction."""
    in_dim, *hidden_dims, out_dim = topology

    inp = keras.Input(shape=(in_dim,))
    h = inp
    for hd in hidden_dims:
        h = keras.layers.Dense(hd, use_bias=True)(h)
        h = keras.layers.Lambda(sign_activation)(h)
    out = keras.layers.Dense(out_dim, use_bias=True)(h)

    m = keras.Model(inp, out)
    m.set_weights(weights)
    return m

eval_model = build_strict_sign_eval(model.get_weights(), CONFIG["topology"])
eval_model.compile(loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])

loss, accuracy = eval_model.evaluate(x_test_bin, y_test, verbose=0)
print(f"Cleartext accuracy after discretization (strict sign): {accuracy * 100:.2f}%   (chance = 10.00%)")


Cleartext accuracy after discretization (strict sign): 57.63%   (chance = 10.00%)


## 8. Export integer weights + companion JSON config

**Weights:** `2 · N` headerless CSVs written with `fmt='%d'`. Filenames are `<prefix>_W{i}.csv` / `<prefix>_b{i}.csv`, matching the convention the C++ sub-projects (`MNIST_30/`, `MNIST_100/`, `cifar10/`) already read.

**Config:** a single companion `<prefix>_config.json` that captures the topology, training/discretization hyperparameters, the cleartext discretized accuracy, and the per-layer L1 / L2 norms — `max(L1)` of `Wi` is the OpenFHE message-space bound `B` you should size the plaintext modulus around; `max(L2)` is the noise-growth factor at that layer.

In [46]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
prefix = CONFIG["prefix"]

csv_paths: list[Path] = []
for i, (W, b) in enumerate(zip(Ws_int, bs_int), start=1):
    pW = OUTPUT_DIR / f"{prefix}_W{i}.csv"
    pb = OUTPUT_DIR / f"{prefix}_b{i}.csv"
    np.savetxt(pW, W, delimiter=",", fmt="%d")
    np.savetxt(pb, b,                 fmt="%d")
    csv_paths += [pW, pb]

# Per-layer L1 / L2 column norms on the integer weights.
# axis=0 sums down the columns: each column is the input vector to one neuron.
openfhe_layers = []
for i, W in enumerate(Ws_int, start=1):
    l1 = np.sum(np.abs(W), axis=0)
    l2 = np.linalg.norm(W, axis=0)
    openfhe_layers.append({
        "layer":      i,
        "max_l1":    int(l1.max()),
        "max_l2":    float(l2.max()),
        "mean_l1":   float(l1.mean()),
        "mean_l2":   float(l2.mean()),
    })
    print(f"  Layer {i}: max L1 = {int(l1.max()):>6d}   max L2 = {l2.max():.2f}   "
          f"(mean L1 = {l1.mean():.1f}, mean L2 = {l2.mean():.2f})")

B_recommended = max(layer["max_l1"] for layer in openfhe_layers)
print()
print(f"--- OpenFHE Parameters ---")
print(f"Recommended message-space bound B >= {B_recommended} "
      f"(max L1 across all Linear layers).")

config_path = OUTPUT_DIR / f"{prefix}_config.json"
run_config = {
    "dataset":          CONFIG["dataset"],
    "approach":         "keras-continuous-then-posthoc-discretize",
    "prefix":           prefix,
    "output_dir":       str(OUTPUT_DIR),
    "model": {
        "topology":             list(CONFIG["topology"]),
        "training_activation":  CONFIG["hidden_activation"],
        "eval_activation":      "sign",
        "tau":                  int(CONFIG["tau"]),
    },
    "training": {
        "epochs":           int(CONFIG["epochs"]),
        "batch_size":       int(CONFIG["batch_size"]),
        "validation_split": float(CONFIG["validation_split"]),
        "binarize_threshold": int(CONFIG["binarize_threshold"]),
        "seed":             int(CONFIG["seed"]),
        "optimizer":        "adam",
        "loss":             "sparse_categorical_crossentropy",
    },
    "results": {
        "final_train_acc": float(history.history["accuracy"][-1]),
        "final_train_loss": float(history.history["loss"][-1]),
        "final_val_acc":   float(history.history.get("val_accuracy", [float('nan')])[-1]),
        "final_val_loss":  float(history.history.get("val_loss",     [float('nan')])[-1]),
        "discretized_test_acc":  float(accuracy),
        "discretized_test_loss": float(loss),
    },
    "openfhe_params": {
        "B_recommended": int(B_recommended),
        "per_layer":     openfhe_layers,
    },
    "weights": {
        "fmt":   "%d",
        "files": [p.name for p in csv_paths],
    },
    "tensorflow_version": tf.__version__,
    "timestamp":          datetime.datetime.now().isoformat(timespec="seconds"),
}

with open(config_path, "w") as f:
    _json.dump(run_config, f, indent=2)

print()
for p in csv_paths:
    print(f"  {p.name:30s}  ({p.stat().st_size:>7d} bytes)")
print(f"  {config_path.name:30s}  ({config_path.stat().st_size:>7d} bytes)  <-- run config")


  Layer 1: max L1 =    814   max L2 = 45.12   (mean L1 = 710.8, mean L2 = 38.02)
  Layer 2: max L1 =    347   max L2 = 42.88   (mean L1 = 301.7, mean L2 = 38.21)
  Layer 3: max L1 =    486   max L2 = 55.48   (mean L1 = 445.9, mean L2 = 52.22)

--- OpenFHE Parameters ---
Recommended message-space bound B >= 814 (max L1 across all Linear layers).

  mnist_W1.csv                    ( 179521 bytes)
  mnist_b1.csv                    (    203 bytes)
  mnist_W2.csv                    (  24581 bytes)
  mnist_b2.csv                    (    208 bytes)
  mnist_W3.csv                    (   2524 bytes)
  mnist_b3.csv                    (     21 bytes)
  mnist_config.json               (   1633 bytes)  <-- run config


## 9. Summary

The artifact directory now contains:

* `<prefix>_W{i}.csv`, `<prefix>_b{i}.csv` — `2 · N` headerless integer files for an `N`-Dense topology, ready for the C++ `io::LoadCsv2D` / `io::LoadCsv1D` loaders.
* `<prefix>_config.json` — full run summary (topology, activation, `τ`, training hyper-parameters, accuracy, OpenFHE per-layer L1 / L2 norms).

Wire the CSVs into a sub-project the same way as `MNIST_30/main.cpp` already does:
```cpp
auto W1 = io::LoadCsv2D("../<prefix>_W1.csv", IN_DIM, HID_DIM);
auto b1 = io::LoadCsv1D("../<prefix>_b1.csv");
// ... one pair per Dense layer ...
```
and configure OpenFHE with a plaintext modulus `>= openfhe_params.B_recommended` from the JSON config above.